# 🎬 Promo Extractor (Kuku · Problem 1)
Finds 20-60s promo clips in a microdrama screenplay: hook in the first 3s, cliffhanger at the cut, makes sense to a stranger.

**Pipeline:** parse (Python) → Hook Hunter (LLM, 1 call/episode) → 60s window (Python) → Judge & Trimmer (LLM, 1 call/clip) + timestamp checks (Python) → tiers + Ranker (LLM, 1 call) → hand-check sheet.

**Before you start:** add your Gemini key in the 🔑 *Secrets* panel (left sidebar) as `GEMINI_API_KEY`, and turn on *Notebook access*.

## 1. Get the code

In [ ]:
REPO = "https://github.com/ShubNoob/KUKU_promo"   # <- your repo

import glob, os
%cd /content
!rm -rf repo && git clone -q $REPO repo
# find the folder that holds run.py, wherever it sits in the repo
hits = glob.glob("repo/**/run.py", recursive=True)
assert hits, "run.py not found in the repo - check that all files were uploaded to GitHub"
%cd {os.path.dirname(os.path.abspath(hits[0]))}
!ls
!pip -q install google-genai pydantic pyyaml

## 2. API key + check which models your key can use

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from google import genai
client = genai.Client()
print([m.name for m in client.models.list() if "flash" in m.name])

**Swap the model here** (or edit `config.yaml`). Any Gemini model ID from the list above works.
To use another provider (Groq, OpenRouter, OpenAI, Ollama), set `provider: openai_compatible`, `base_url` and `api_key_env` in `config.yaml`.

In [ ]:
MODEL = "gemini-flash-latest"

## 3. Upload the episode scripts (.zip of .txt files)

In [ ]:
from google.colab import files
up = files.upload()
ZIP = next(iter(up))
print("Uploaded:", ZIP)

## 4. Sanity checks (no API calls)
Unit tests for the timestamp checks, then an offline dry run with the keyword *mock* model.

In [ ]:
!python tests/test_checks.py
!python run.py "$ZIP" --provider mock --out outputs_mock | tail -4

## 5. Quick real run on 3 episodes

In [ ]:
!python run.py "$ZIP" --model $MODEL --episodes 1-3 --out outputs_test

## 6. Full run on every episode
Responses are cached in `.cache/`, so re-running after an interruption only pays for the calls that didn't finish.

In [ ]:
!python run.py "$ZIP" --model $MODEL --out outputs

## 7. Results

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("outputs/shortlist.md").read()))

In [ ]:
import pandas as pd
pd.read_csv("outputs/promos.csv").head(20)

## 8. Download the raw outputs

In [ ]:
!zip -qr raw_outputs.zip outputs
files.download("raw_outputs.zip")

## 9. Hand check (human in the loop)
Open `outputs/hand_check.csv`, watch/read each clip, fill `human_good` (y/n) and `human_reason`, upload it back and score it.
The sheet hides the model's scores so you judge first.

In [ ]:
files.download("outputs/hand_check.csv")

In [ ]:
labelled = next(iter(files.upload()))
!python run.py --score "$labelled"